# scVI experiment

In [ ]:
if False:
import scvi

# adata = ad.read_h5ad('../output/temp/CD8T_bulk_tf_activity_NN_adata.h5ad')
# adata.X = adata.X+np.min(adata.X)

adata = ad.read_h5ad('../../task_grn_inference/resources/grn_benchmark/inference_data/op_rna.h5ad')

In [ ]:
import torch
from torch import nn
from scvi.modules import VAEModule
from scvi.nn import FCLayers
class SCVIAge(VAEModule):
    def __init__(self, n_input, n_batch, **kwargs):
        super().__init__(
            n_input=n_input,
            n_batch=n_batch,
            n_labels=None,
            **kwargs
        )

        self.age_predictor = FCLayers(
            n_in=self.latent_dim,
            n_out=1,  # regression; for classification, use n_out=n_classes
            n_hidden=64,
            n_layers=2,
            dropout_rate=0.1
        )

        self.age_loss_fn = nn.MSELoss()

    def forward(self, x, batch_index, age=None):
        inference_outputs = self.inference(x, batch_index)
        generative_outputs = self.generative(
            z=inference_outputs["z"],
            library=inference_outputs["library"],
            batch_index=batch_index,
        )

        # Predict age from z
        age_pred = self.age_predictor(inference_outputs["z"])
        inference_outputs["age_pred"] = age_pred
        inference_outputs["age_true"] = age

        return inference_outputs, generative_outputs

    def loss(self, tensors, inference_outputs, generative_outputs):
        loss_dict = super().loss(tensors, inference_outputs, generative_outputs)

        if inference_outputs["age_true"] is not None:
            age = inference_outputs["age_true"].float().unsqueeze(1)
            age_pred = inference_outputs["age_pred"]
            age_loss = self.age_loss_fn(age_pred, age)
            loss_dict["age_loss"] = age_loss
            loss_dict["loss"] += age_loss

        return loss_dict

In [ ]:
# Register AnnData
scvi.model.SCVI.setup_anndata(adata, batch_key="donor_id")

# Create and train model
model = scvi.model.SCVI(adata)
model.to_device("cuda")
model.train()